# Run 3 — one chart at a time: Qwen3.5-9B + LoRA on Quest **videos**

Set **`CHART`** in cell 3 (`"line"` first, then `"pie"`), then run the cells in order (Shift + Enter).
Works on A100 (full settings) and T4 (4-bit, smaller frames, chosen automatically).

Training uses only synthetic clips of made-up charts of that type (`--strict`). Testing uses your real Quest videos
of that chart, 8 frames each, plus the prompt. Your truth tables are used only to score the answers.

| Step | What it does | A100 | T4 |
|---|---|---|---|
| 1 | install Unsloth for Qwen3.5 | 4 min | 4 min |
| 2 | upload `dxr_video_upload.zip` | 1 min | 1 min |
| 2b | (optional) upload earlier results zips to keep one combined table | 1 min | 1 min |
| 3 | settings: which chart, GPU check | – | – |
| 4 | generate synthetic clips of that chart | 3 min | 3 min |
| 5 | baseline: Qwen3.5-9B on your videos of that chart | ~5 min | ~15 min |
| 6 | train the LoRA for that chart | ~25 min | ~60–90 min |
| 7 | test the fine-tuned model | ~5 min | ~15 min |
| 8 | comparison table for that chart | seconds | seconds |
| 9 | download | 1 min | 1 min |

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# 2 · Upload dxr_video_upload.zip (made on your Mac)
from google.colab import files
import zipfile
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall("/content/dxr")
%cd /content/dxr
!cat real_clips/real.jsonl | python3 -c "import sys,json; [print(json.loads(l)['chart'], json.loads(l)['video']) for l in sys.stdin]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content/dxr
# 2b · (optional) upload ft_results.zip and/or earlier video_results_*.zip so all results land in one table
up2 = files.upload()
for name in up2:
    zipfile.ZipFile(name).extractall("/content/dxr")
!ls *.csv

In [ ]:
%cd /content/dxr
# 3 · SETTINGS - change CHART to "pie" for the second round
CHART = "line"
TRAIN = 1000                                  # synthetic training clips of this chart (1000 = 125 steps)
MODEL = "unsloth/Qwen3.5-9B"
import torch
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if gb > 30:                                   # A100 / L4-class: full settings
    EVAL_X, TRAIN_X = "--max-side 640", "--max-side 640"
else:                                         # T4 (16 GB): 4-bit and smaller frames
    EVAL_X, TRAIN_X = "--load-4bit --max-side 512", "--load-4bit --max-side 512 --max-length 3072"
STEPS = TRAIN // 8
print(f"chart={CHART}  GPU memory={gb:.0f} GB  eval: {EVAL_X}  train: {TRAIN_X}  steps={STEPS}")

In [ ]:
%cd /content/dxr
# 4 · Synthetic clips of this chart only (strict: no real names/values)
!python finetune_video/make_video_dataset.py --strict --charts {CHART} --train {TRAIN} --val 30 --out finetune_video/data_{CHART}
import json
from PIL import Image
from IPython.display import display
def strip(paths, base):
    ims = [Image.open(f"{base}/{p}").resize((240, 180)) for p in paths]
    s = Image.new("RGB", (240 * len(ims), 180), "white")
    for i, im in enumerate(ims): s.paste(im, (240 * i, 0))
    return s
t = json.loads(open(f"finetune_video/data_{CHART}/train.jsonl").readline()); print("synthetic training clip"); display(strip(t["frames"], f"finetune_video/data_{CHART}"))
for l in open("real_clips/real.jsonl"):
    r = json.loads(l)
    if r["chart"] == CHART: print("real test video:", r["video"]); display(strip(r["frames"], "real_clips"))

In [ ]:
%cd /content/dxr
# 5 · Baseline: Qwen3.5-9B before fine-tuning, this chart only
!python finetune_video/eval_video.py --model {MODEL} --name qwen35_base --charts {CHART} --best --val finetune_video/data_{CHART}/val.jsonl --val-n 10 {EVAL_X}

In [ ]:
%cd /content/dxr
# 6 · Train the LoRA for this chart
!python finetune_video/train_video_lora.py --base {MODEL} --data finetune_video/data_{CHART} --steps {STEPS} --out lora_qwen35_{CHART} {TRAIN_X}

In [ ]:
%cd /content/dxr
# 6b · keep the adapters in Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/dxr_lora && cp -r lora_qwen35_{CHART} /content/drive/MyDrive/dxr_lora/
!ls /content/drive/MyDrive/dxr_lora

In [ ]:
%cd /content/dxr
# 7 · Test the fine-tuned model (same inputs as step 5)
!python finetune_video/eval_video.py --model lora_qwen35_{CHART} --name qwen35_lora_{CHART} --charts {CHART} --best --val finetune_video/data_{CHART}/val.jsonl --val-n 10 {EVAL_X}

In [ ]:
%cd /content/dxr
# 8 · Comparison for this chart: Qwen3-VL runs (if uploaded) vs Qwen3.5 base vs Qwen3.5 + LoRA
!python finetune_video/eval_video.py --compare --charts {CHART} --out compare_{CHART}.md
from IPython.display import Markdown, display
display(Markdown(open(f"compare_{CHART}.md").read()))

In [ ]:
%cd /content/dxr
# 9 · Download (keep this zip - upload it in step 2b next time so the table keeps growing)
!zip -qr video_results_{CHART}.zip results_video.csv compare_{CHART}.md predictions/video_* lora_qwen35_{CHART}/training_info.json
files.download(f"video_results_{CHART}.zip")

### Next chart
Change `CHART = "pie"` in cell 3 and run cells 3–9 again (same session), or in a new session run 1, 2, 2b
(upload `video_results_line.zip`), then 3–9.

### If a step fails
* **"No such file"**: run `%cd /content/dxr`.
* **Model not found**: set `MODEL = "Qwen/Qwen3.5-9B"` in cell 3.
* **Out of memory** (T4): in cell 3 change 512 to 448 in both lines, or set `MODEL = "unsloth/Qwen3.5-4B"`.